In [11]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

In [12]:
# Load Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()  

# Load your B2 JSON data
df_b2 = pd.read_json("b2_welsh.json")
df_b2["cefr_level"] = "B2"
df_b2 = df_b2.drop_duplicates(subset="text", keep="first")

# Combine both DataFrames
df_combined = pd.concat([df_main, df_b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [13]:
ds_merged

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 2020
})

In [14]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [15]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [16]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [17]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [18]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [20]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_b2/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 404/404 [00:00<00:00, 20766.43 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_31800\2938839092.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.186200,1.412899,0.477723,0.378964,0.315882,0.477723,0.549801,0.901961,0.683168,0.359477,0.454545,0.401460,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.793200,0.622060,0.752475,0.751261,0.754337,0.752475,0.778443,0.849673,0.812500,0.682540,0.710744,0.696356,0.000000,0.000000,0.000000,0.792793,0.676923,0.730290,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.395400,0.492552,0.806931,0.804557,0.808067,0.806931,0.796610,0.921569,0.854545,0.800000,0.727273,0.761905,0.000000,0.000000,0.000000,0.829060,0.746154,0.785425,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



 Running Fold 2...


...

Map: 100%|██████████| 404/404 [00:00<00:00, 14268.26 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_31800\2938839092.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.109300,0.851829,0.628713,0.613656,0.720116,0.628713,0.757962,0.782895,0.770227,0.467980,0.785124,0.586420,0.000000,0.000000,0.000000,0.909091,0.305344,0.457143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.735000,0.615217,0.740099,0.732847,0.798239,0.740099,0.810651,0.901316,0.853583,0.581395,0.826446,0.682594,0.000000,0.000000,0.000000,0.984127,0.473282,0.639175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.342900,0.418221,0.826733,0.828733,0.836037,0.826733,0.925170,0.894737,0.909699,0.697183,0.818182,0.752852,0.000000,0.000000,0.000000,0.860870,0.755725,0.804878,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 404/404 [00:00<00:00, 14236.38 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_31800\2938839092.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.127000,1.078430,0.376238,0.293129,0.723578,0.376238,0.805556,0.190789,0.308511,0.321330,0.958678,0.481328,0.000000,0.000000,0.000000,1.000000,0.053435,0.101449,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.775700,0.726811,0.690594,0.672718,0.687463,0.690594,0.792683,0.855263,0.822785,0.632353,0.355372,0.455026,0.000000,0.000000,0.000000,0.616279,0.809160,0.699670,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.439800,0.583894,0.772277,0.773890,0.777027,0.772277,0.864865,0.842105,0.853333,0.654135,0.719008,0.685039,0.000000,0.000000,0.000000,0.788618,0.740458,0.763780,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [21]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_b2/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [22]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [23]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.808067  0.806931  0.804557  0.796610  0.921569  0.854545   
1        2        0.856117  0.856436  0.855175  0.866667  0.934641  0.899371   
2        3        0.771763  0.772277  0.767717  0.774011  0.901316  0.832827   
3        4        0.836037  0.826733  0.828733  0.925170  0.894737  0.909699   
4        5        0.777027  0.772277  0.773890  0.864865  0.842105  0.853333   
5  Average        0.809802  0.806931  0.806015  0.845465  0.898873  0.869955   

         A2                      ...   B1        B2                      \
  Precision    Recall        F1  ...   F1 Precision    Recall        F1   
0  0.800000  0.727273  0.761905  ...  0.0  0.829060  0.746154  0.785425   
1  0.842975  0.842975  0.842975  ...  0.0  0.855932  0.776923  0.814516   
2  0.765306  0.614754  0.681818  ...  0.0  0.775194  0.769231  0.772201   
3  0.697183  0.818182  0.752852  ...  0.0  0.860870  0.755725  0.804878   
4  0.654135  0.719008  0.685039  ...  0.0  0.788618  0.740458  0.763780   
5  0.751920  0.744438  0.744918  ...  0.0  0.821935  0.757698  0.788160   

         C1                    C2              
  Precision Recall   F1 Precision Recall   F1  
0       0.0    0.0  0.0       0.0    0.0  0.0  
1       0.0    0.0  0.0       0.0    0.0  0.0  
2       0.0    0.0  0.0       0.0    0.0  0.0  
3       0.0    0.0  0.0       0.0    0.0  0.0  
4       0.0    0.0  0.0       0.0    0.0  0.0  
5       0.0    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]